# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bilalahmed251/-ML-Search-Discovery/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [12]:
# This cell is for CODE (numbers, a query, a check).
# I will prioritize pages that are visible but appear to have improvement opportunity. The baseline score will combine high impressions, older content, weaker average position, and lower CTR. The score is a review-priority signal, not a prediction or a guarantee.

# Reason codes:
#- HIGH_VISIBILITY: the page has relatively high impressions.
#- OLD_CONTENT: the page is relatively old or has not been updated recently.
#- WEAK_POSITION: the page has a relatively weak average search position.
#- LOW_CTR: the page has relatively low CTR compared with other visible pages.



In [13]:
import os
import pandas as pd
import numpy as np

repo = "/content/ML-Search-Discovery"

if not os.path.exists(repo):
    !git clone -q --depth 1 https://github.com/bilalahmed251/-ML-Search-Discovery.git {repo}

df = pd.read_csv(
    f"{repo}/data/raw/content_refresh_anonymized.csv"
 ).copy()

print("Rows:", len(df))
print("Columns:", len(df.columns))


Rows: 30000
Columns: 44


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [14]:
# This cell is for CODE (numbers, a query, a check).
# I will create a transparent baseline score using only information available for the page. I will rank every page from highest to lowest score and save the complete review queue as baseline_action_score.csv. The score will not use trend_direction or trend_pct because those fields describe the outcome and could create leakage.



In [15]:
# Required numeric fields
required = [
    "content_id",
    "impressions_90d",
    "ctr",
    "avg_position",
    "content_age_days",
    "days_since_last_update",
]

missing = [col for col in required if col not in df.columns]
print("Missing required fields:", missing)

# Work only with available rows for the score
queue = df[required].copy()

for col in required[1:]:
    queue[col] = pd.to_numeric(queue[col], errors="coerce")

queue = queue.dropna().copy()

# Percentile ranks:
# high impressions = more visible
# high age/update delay = more refresh opportunity
# high average position number = weaker ranking
# low CTR = more opportunity
queue["visibility_score"] = queue["impressions_90d"].rank(pct=True)
queue["age_score"] = queue["content_age_days"].rank(pct=True)
queue["stale_score"] = queue["days_since_last_update"].rank(pct=True)
queue["position_score"] = queue["avg_position"].rank(pct=True)
queue["low_ctr_score"] = 1 - queue["ctr"].rank(pct=True)

# Transparent hand-written score
queue["baseline_score"] = (
    0.30 * queue["visibility_score"]
    + 0.20 * queue["age_score"]
    + 0.20 * queue["stale_score"]
    + 0.15 * queue["position_score"]
    + 0.15 * queue["low_ctr_score"]
)

# Reason codes
def reason_codes(row):
    reasons = []

    if row["visibility_score"] >= 0.75:
        reasons.append("HIGH_VISIBILITY")

    if row["age_score"] >= 0.75 or row["stale_score"] >= 0.75:
        reasons.append("OLD_CONTENT")

    if row["position_score"] >= 0.75:
        reasons.append("WEAK_POSITION")

    if row["low_ctr_score"] >= 0.75:
        reasons.append("LOW_CTR")

    return "|".join(reasons) if reasons else "MIXED_SIGNAL"

queue["reason_code"] = queue.apply(reason_codes, axis=1)

# Rank from highest to lowest priority
queue = queue.sort_values(
    "baseline_score",
    ascending=False
).reset_index(drop=True)

queue.insert(0, "priority_rank", range(1, len(queue) + 1))

# Save the complete ranked queue
output_dir = f"{repo}/work/outputs"
os.makedirs(output_dir, exist_ok=True)

output_path = f"{output_dir}/baseline_action_score.csv"
queue.to_csv(output_path, index=False)

print("Saved:", output_path)
print("Ranked rows:", len(queue))
display(queue.head(20))


Missing required fields: []
Saved: /content/ML-Search-Discovery/work/outputs/baseline_action_score.csv
Ranked rows: 30000


,priority_rank,content_id,impressions_90d,ctr,avg_position,content_age_days,days_since_last_update,visibility_score,age_score,stale_score,position_score,low_ctr_score,baseline_score,reason_code
0,1,content_8d0a8cbf9d1e,5090,0.00,47.7,537,104,0.797333,0.987917,0.843200,0.948900,0.779783,0.864726,HIGH_VISIBILITY|OLD_CONTENT|WEAK_POSITION|LOW_CTR
1,2,content_62abc4bd66be,31364,0.09,68.5,445,104,0.966083,0.866900,0.843200,0.988333,0.472150,0.850917,HIGH_VISIBILITY|OLD_CONTENT|WEAK_POSITION
2,3,content_fb4bf6555c79,84093,0.00,45.6,299,104,0.992367,0.619517,0.843200,0.942083,0.779783,0.848533,HIGH_VISIBILITY|OLD_CONTENT|WEAK_POSITION|LOW_CTR
3,4,content_1f3b8e699416,59474,0.06,35.2,445,104,0.986233,0.866900,0.843200,0.889267,0.512833,0.848205,HIGH_VISIBILITY|OLD_CONTENT|WEAK_POSITION
4,5,content_a0d9819c4569,31783,0.06,41.6,445,104,0.966867,0.866900,0.843200,0.925083,0.512833,0.847768,HIGH_VISIBILITY|OLD_CONTENT|WEAK_POSITION
5,6,content_d376881f5bd0,5301,0.00,54.8,445,104,0.803583,0.866900,0.843200,0.967867,0.779783,0.845243,HIGH_VISIBILITY|OLD_CONTENT|WEAK_POSITION|LOW_CTR
6,7,content_9ecd60ff3bca,4543,0.00,71.3,445,104,0.782350,0.866900,0.843200,0.991200,0.779783,0.842373,HIGH_VISIBILITY|OLD_CONTENT|WEAK_POSITION|LOW_CTR
7,8,content_05133844bff4,14803,0.00,54.4,482,22,0.917433,0.934717,0.588283,0.967083,0.779783,0.841860,HIGH_VISIBILITY|OLD_CONTENT|WEAK_POSITION|LOW_CTR
8,9,content_479f8ec909fe,35429,0.04,29.8,445,104,0.971733,0.866900,0.843200,0.843867,0.535867,0.840500,HIGH_VISIBILITY|OLD_CONTENT|WEAK_POSITION
9,10,content_5b328cf43d53,3144,0.00,39.0,545,104,0.730550,0.993600,0.843200,0.912617,0.779783,0.840385,OLD_CONTENT|WEAK_POSITION|LOW_CTR


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [16]:
# This cell is for CODE (numbers, a query, a check).
# The top 20 pages are a review queue, not automatic recommendations. For each page, I will record the proposed action, the reason code, a confidence note, and what could make the recommendation wrong. A page may be a weak pick if it has low traffic, incomplete history, unusual seasonality, or a measurement issue.


In [17]:
top20 = queue.head(20).copy()

top20["action"] = "MANUAL_REVIEW"

top20["confidence_note"] = np.where(
    top20["impressions_90d"] >= 1000,
    "Higher confidence: sufficient visibility",
    "Lower confidence: check traffic volume and history"
)

top20["what_could_make_it_wrong"] = (
    "Seasonality, tracking issue, recent update, or insufficient history"
)

review_columns = [
    "priority_rank",
    "content_id",
    "baseline_score",
    "action",
    "reason_code",
    "confidence_note",
    "what_could_make_it_wrong",
    "impressions_90d",
    "ctr",
    "avg_position",
    "content_age_days",
    "days_since_last_update",
]

top20_review = top20[review_columns].copy()

display(top20_review)

review_path = f"{output_dir}/top20_review.csv"
top20_review.to_csv(review_path, index=False)

print("Saved:", review_path)


,priority_rank,content_id,baseline_score,action,reason_code,confidence_note,what_could_make_it_wrong,impressions_90d,ctr,avg_position,content_age_days,days_since_last_update
0,1,content_8d0a8cbf9d1e,0.864726,MANUAL_REVIEW,HIGH_VISIBILITY|OLD_CONTENT|WEAK_POSITION|LOW_CTR,Higher confidence: sufficient visibility,"Seasonality, tracking issue, recent update, or...",5090,0.00,47.7,537,104
1,2,content_62abc4bd66be,0.850917,MANUAL_REVIEW,HIGH_VISIBILITY|OLD_CONTENT|WEAK_POSITION,Higher confidence: sufficient visibility,"Seasonality, tracking issue, recent update, or...",31364,0.09,68.5,445,104
2,3,content_fb4bf6555c79,0.848533,MANUAL_REVIEW,HIGH_VISIBILITY|OLD_CONTENT|WEAK_POSITION|LOW_CTR,Higher confidence: sufficient visibility,"Seasonality, tracking issue, recent update, or...",84093,0.00,45.6,299,104
3,4,content_1f3b8e699416,0.848205,MANUAL_REVIEW,HIGH_VISIBILITY|OLD_CONTENT|WEAK_POSITION,Higher confidence: sufficient visibility,"Seasonality, tracking issue, recent update, or...",59474,0.06,35.2,445,104
4,5,content_a0d9819c4569,0.847768,MANUAL_REVIEW,HIGH_VISIBILITY|OLD_CONTENT|WEAK_POSITION,Higher confidence: sufficient visibility,"Seasonality, tracking issue, recent update, or...",31783,0.06,41.6,445,104
5,6,content_d376881f5bd0,0.845243,MANUAL_REVIEW,HIGH_VISIBILITY|OLD_CONTENT|WEAK_POSITION|LOW_CTR,Higher confidence: sufficient visibility,"Seasonality, tracking issue, recent update, or...",5301,0.00,54.8,445,104
6,7,content_9ecd60ff3bca,0.842373,MANUAL_REVIEW,HIGH_VISIBILITY|OLD_CONTENT|WEAK_POSITION|LOW_CTR,Higher confidence: sufficient visibility,"Seasonality, tracking issue, recent update, or...",4543,0.00,71.3,445,104
7,8,content_05133844bff4,0.841860,MANUAL_REVIEW,HIGH_VISIBILITY|OLD_CONTENT|WEAK_POSITION|LOW_CTR,Higher confidence: sufficient visibility,"Seasonality, tracking issue, recent update, or...",14803,0.00,54.4,482,22
8,9,content_479f8ec909fe,0.840500,MANUAL_REVIEW,HIGH_VISIBILITY|OLD_CONTENT|WEAK_POSITION,Higher confidence: sufficient visibility,"Seasonality, tracking issue, recent update, or...",35429,0.04,29.8,445,104
9,10,content_5b328cf43d53,0.840385,MANUAL_REVIEW,OLD_CONTENT|WEAK_POSITION|LOW_CTR,Higher confidence: sufficient visibility,"Seasonality, tracking issue, recent update, or...",3144,0.00,39.0,545,104


Saved: /content/ML-Search-Discovery/work/outputs/top20_review.csv


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [18]:
# This cell is for CODE (numbers, a query, a check).
# Some high-ranked pages may be weak picks because a high score does not prove that a refresh will improve performance. Possible problems include low traffic, incomplete history, seasonality, tracking errors, or a page that was recently changed. I will not use trend_direction or trend_pct in the baseline score because they describe the outcome and may leak future or target information. The queue is therefore directional decision support, not causal proof.



In [19]:
# Show possible weak picks among the top 20
weak_picks = top20[
    (top20["impressions_90d"] < 100) |
    (top20["ctr"] < 0) |
    (top20["avg_position"] <= 0)
].copy()

print("Possible weak picks in top 20:", len(weak_picks))

if len(weak_picks) > 0:
    display(weak_picks[review_columns])
else:
    print("No obvious weak picks found using the basic checks.")

# Leakage check
score_columns = [
    "visibility_score",
    "age_score",
    "stale_score",
    "position_score",
    "low_ctr_score",
    "baseline_score",
]

leakage_fields = [
    "trend_direction",
    "trend_pct",
    "is_declining_label",
]

used_leakage_fields = [
    field for field in leakage_fields
    if field in queue.columns
]

print("\nFields used in score:")
print(score_columns)

print("\nLeakage fields used in score:")
print(used_leakage_fields)

print("\nLeakage check passed:", len(used_leakage_fields) == 0)


Possible weak picks in top 20: 0
No obvious weak picks found using the basic checks.

Fields used in score:
['visibility_score', 'age_score', 'stale_score', 'position_score', 'low_ctr_score', 'baseline_score']

Leakage fields used in score:
[]

Leakage check passed: True


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.